In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, explained_variance_score
from sklearn.model_selection import GridSearchCV
from matplotlib import pyplot as plt

In [3]:
rounding = 3

In [4]:
def print_feature_importances(cols, importances):
    idx = np.argsort(importances)[::-1]
    print(list(zip(np.array(cols)[idx], np.array(importances)[idx])))

def feature_importance_dict(cols, importances):
    dict = {}
    for i in range(len(cols)):
        col = cols[i]
        dict[col] = np.round(importances[i], rounding)
    return dict

In [6]:
df = pd.read_csv('../processed_data/processed_data.csv')
basin_dict = {"AL": 0.0, "CP": 1.0, "EP": 2.0}
for i in range(len(df)):
    df.at[i, "Basin"] = basin_dict[df.iloc[i]["Basin"]]
df["Basin"]

0       0.0
1       0.0
2       0.0
3       0.0
4       0.0
       ... 
1121    2.0
1122    2.0
1123    2.0
1124    2.0
1125    2.0
Name: Basin, Length: 1126, dtype: object

In [9]:
cols = ["logVMAX12", "MSLP12", "POT12", "VGRAD0-6", "VGRAD6-12", "ONI", "AL", "CP", "EP"]
X, y = df[cols].to_numpy(), df["logVMAX36"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=4)

rf = RandomForestRegressor()
# Grid search
params = {"n_estimators": [20, 50, 100],
          "criterion": ["squared_error", "friedman_mse", "absolute_error", "poisson"],
          "max_depth": [None, 3, 5, 10],
          "min_samples_split": [2, 3, 5, 8],
          "max_features": [None, 2, 3, 5]}
rf_gs = GridSearchCV(rf, params, scoring="explained_variance", n_jobs=-1)

rf_gs.fit(X_train, y_train)

print("Best params:", rf_gs.best_params_)
print("Best score:", rf_gs.best_score_)

KeyError: "['AL', 'CP', 'EP'] not in index"

In [8]:
rf = RandomForestRegressor(**rf_gs.best_params_)
fit_dtr = rf.fit(X_train, y_train)
rf_importances_df = pd.DataFrame({"Importance": feature_importance_dict(cols, fit_dtr.feature_importances_)}).T
print(rf_importances_df.to_latex(caption="Feature Importances of sklearn Random Forest Regressor with Basin Indicator", label="tab:rf_importances"))
rf_importances_df

\begin{table}
\caption{Feature Importances of sklearn Random Forest Regressor with Basin Indicator}
\label{tab:rf_importances}
\begin{tabular}{lrrrrrrr}
\toprule
 & logVMAX12 & MSLP12 & POT12 & VGRAD0-6 & VGRAD6-12 & ONI & Basin \\
\midrule
Importance & 0.408000 & 0.151000 & 0.073000 & 0.048000 & 0.244000 & 0.048000 & 0.027000 \\
\bottomrule
\end{tabular}
\end{table}



,logVMAX12,MSLP12,POT12,VGRAD0-6,VGRAD6-12,ONI,Basin
Importance,0.408,0.151,0.073,0.048,0.244,0.048,0.027
